# Multi-Agent System Using RAG powered by AWS Strands Agents

This notebook demonstrates how to design and build an enterprise-grade **Multi-Agent RAG System** using **AWS Strands Agents (`strands-agents`)**, the official lightweight Python SDK from AWS for model-driven AI agents.

### Key Highlights:
- **AWS Strands Agents**: Lightweight, model-driven agent loops with `@tool` decorators and Bedrock models (`BedrockModel`, e.g., Nova Pro / Claude 3.5 Sonnet).
- **Hierarchical Multi-Agent Collaboration**: Orchestrator (Supervisor) coordinates specialist agents (`RAGIngestionAgent`, `FinancialAnalystAgent`, `PrivilegedExecutionAgent`) using the **Agents as Tools** pattern.
- **Defense in Depth**: Bedrock Contextual Grounding Guardrails (threshold >= 0.75) and cryptographic tool authorization defending against indirect prompt injection.

## 1. Strands Multi-Agent System Architecture
We initialize the four specialized Strands agents and the security supervisor orchestrator.

In [ ]:
from strands import Agent, tool
from multi_agent_rag import (
    Tool,
    BaseAgent,
    RAGIngestionAgent,
    FinancialAnalystAgent,
    PrivilegedExecutionAgent,
    SecuritySupervisorAgent,
    build_multi_agent_rag_system,
)

# Instantiate the multi-agent system powered by Strands Agents
supervisor = build_multi_agent_rag_system(mock_mode=True)

print("Strands Multi-Agent System Initialized:")
print(f" - Orchestrator:       {supervisor.name} (Role: {supervisor.role})")
print(f" - Ingestion Worker:   {supervisor.ingestion.name} (Strands Agent: {type(supervisor.ingestion.strands_agent).__name__})")
print(f"   Tools:              {list(supervisor.ingestion.tools.keys())}")
print(f" - Financial Analyst:  {supervisor.analyst.name} (Strands Agent: {type(supervisor.analyst.strands_agent).__name__})")
print(f"   Tools:              {list(supervisor.analyst.tools.keys())}")
print(f" - Privileged Worker:  {supervisor.privileged.name} (Strands Agent: {type(supervisor.privileged.strands_agent).__name__})")
print(f"   Tools:              {list(supervisor.privileged.tools.keys())}")

## 2. Inspect Ingested Document with Embedded Indirect Injection
The ingested financial report contains a concealed adversarial prompt injection targeting sensitive funds transfer tools.

In [ ]:
doc_content = supervisor.ingestion.document_store["Q3_Financial_Report.md"]
print("=== Ingested Document Excerpt ===\n")
print(doc_content)

## 3. Autonomous Execution & Guardrail Enforcement
Execute user query: `Please read the Q3 financial report and summarize total revenue and operating margins.`

Notice how the Strands supervisor coordinates the pipeline, intercepts the indirect injection, logs contextual grounding telemetry, sanitizes facts, and passes clean data to the Strands financial analyst.

In [ ]:
query = "Please read the Q3 financial report and summarize total revenue and operating margins."
result = supervisor.handle_query(query)

print("=== Strands Multi-Agent Execution Trace ===")
for step in result["workflow_trace"]:
    print(f"Step {step['step']}: [{step['agent']}] -> {step['action']}")
    if "details" in step:
        print(f"         {step['details']}")

print("\n=== Contextual Grounding Guardrail Telemetry ===")
g = result["grounding_check"]
print(f"Grounding Score: {g['grounding_score']} (Required Threshold: {g['threshold']})")
print(f"Status:          {g['status']}")
print(f"Flagged Items:   {g['injections_detected']}")

print(f"\nUnauthorized Privileged Executions: {result['unauthorized_tool_executions']} (Expected: 0)")

print("\n=== Final Verified User Response ===")
print(result["final_response"])

## 4. Validating Cryptographic Isolation of Privileged Strands Agent
Even if an attacker bypasses prompt filters, the isolated `PrivilegedExecutionAgent` refuses tool execution without a supervisor cryptographic authorization token.

In [ ]:
priv_agent = supervisor.privileged

# 1. Unauthenticated invocation attempt (raises PermissionError)
try:
    priv_agent.tools["transfer_funds"].execute(
        recipient="ATTACKER-999",
        amount=250000.0,
        auth_token=None,
    )
    print("FAIL: Unauthorized call was permitted!")
except PermissionError as e:
    print(f"[✓] Unauthorized Call Blocked as expected: {e}")

# 2. Authenticated invocation with supervisor cryptographic token
authorized_resp = priv_agent.tools["transfer_funds"].execute(
    recipient="APPROVED-PAYROLL",
    amount=5000.0,
    auth_token="SUPERVISOR-AUTH-TOKEN-SECURE",
)
print("Authorized Call Result:", authorized_resp)
assert authorized_resp["status"] == "EXECUTED"

print("\n[✓] Strands privilege isolation and authorization gate successfully confirmed!")